In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------------------------------------------------
# 1. Cargar dataset limpio
# ----------------------------------------------------------

ruta_dataset_limpio = Path('../data/2020-Apr-L.csv')

df_ml = pd.read_csv(
    ruta_dataset_limpio,
    parse_dates=['event_time']
)

print(df_ml['event_time'].dtype)
df_ml.head()

datetime64[us, UTC]


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2020-04-01 00:00:00+00:00,view,1201465,2232732101407408685,apparel.shoes.slipons,samsung,230.38,568984877,e2456cef-2d4f-42b9-a53a-8893cb0c6851
1,2020-04-01 00:00:03+00:00,view,9500109,2232732104175649385,apparel.scarf,defender,25.05,530206135,e3c1fb4b-0a7e-457d-a0cf-5d1479e9aafc
2,2020-04-01 00:00:05+00:00,view,1201295,2232732101407408685,apparel.shoes.slipons,apple,489.05,635165586,48d05455-e287-4c44-84f9-76621e02b612
3,2020-04-01 00:00:09+00:00,view,9200640,2232732104343421549,apparel.scarf,defender,20.19,533896443,6a220235-f4d6-4987-a51e-8f315b3027fc
4,2020-04-01 00:00:11+00:00,view,17301057,2232732098446229999,apparel.shoes.sandals,unknown,31.18,543165860,7eaf3210-1d5e-4abc-8437-ae54960c3570


In [4]:
# ==========================================================
# PESOS DE INTERACCIÓN Y SCORE USUARIO-PRODUCTO
# ==========================================================

import numpy as np

pesos_evento = {
    'view': 1,
    'cart': 2,
    'purchase': 3
}

df_ml['peso_evento'] = df_ml['event_type'].map(pesos_evento)

df_score = df_ml.groupby(
    ['user_id', 'product_id'],
    as_index=False
).agg(
    peso_total=('peso_evento', 'sum'),
    frecuencia=('event_type', 'count'),
    ultima_interaccion=('event_time', 'max')
)

fecha_referencia = df_ml['event_time'].max()

df_score['dias_desde_ultima_interaccion'] = (
    fecha_referencia - df_score['ultima_interaccion']
).dt.days

df_score['recencia'] = 1 / (
    1 + df_score['dias_desde_ultima_interaccion']
)

# Score controlado
df_score['score'] = (
    df_score['peso_total'] *
    np.log1p(df_score['frecuencia']) *
    df_score['recencia']
)

df_score.head()

,user_id,product_id,peso_total,frecuencia,ultima_interaccion,dias_desde_ultima_interaccion,recencia,score
0,29515875,1201254,1,1,2020-04-23 05:03:44+00:00,7,0.125,0.086643
1,29515875,1201256,1,1,2020-04-23 04:54:40+00:00,7,0.125,0.086643
2,29515875,1201421,1,1,2020-04-23 05:01:43+00:00,7,0.125,0.086643
3,29515875,1201564,1,1,2020-04-23 04:59:01+00:00,7,0.125,0.086643
4,29515875,100044764,1,1,2020-04-23 04:50:24+00:00,7,0.125,0.086643


In [5]:
# ==========================================================
# NORMALIZACIÓN DEL SCORE
# ==========================================================

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df_score['score_norm'] = scaler.fit_transform(
    df_score[['score']]
)

df_score.sort_values(
    by='score_norm',
    ascending=False
).head()

,user_id,product_id,peso_total,frecuencia,ultima_interaccion,dias_desde_ultima_interaccion,recencia,score,score_norm
2702733,582249321,17303060,724,421,2020-04-30 15:55:37+00:00,0,1.0,4376.583847,1.000000
527123,515145806,100189460,331,189,2020-04-30 16:28:44+00:00,0,1.0,1736.764968,0.396828
876997,518779617,100190575,312,182,2020-04-30 07:15:45+00:00,0,1.0,1625.359680,0.371373
2192045,562225762,100190575,291,173,2020-04-30 12:23:47+00:00,0,1.0,1501.285092,0.343023
2896736,590176847,100190575,274,210,2020-04-30 19:13:25+00:00,0,1.0,1466.409129,0.335054


In [6]:
# ==========================================================
# MATRIZ USUARIO-PRODUCTO EN FORMATO DISPERSO CSR
# Para filtrado colaborativo con KNN
# ==========================================================

from scipy.sparse import csr_matrix

print("\nConstruyendo la Matriz Usuario-Producto...")

usuarios_unicos = df_score['user_id'].unique()
productos_unicos = df_score['product_id'].unique()

# Mapear IDs reales a índices numéricos
user_to_index = {
    user_id: idx for idx, user_id in enumerate(usuarios_unicos)
}

product_to_index = {
    product_id: idx for idx, product_id in enumerate(productos_unicos)
}

# Crear índices de filas y columnas
filas = df_score['user_id'].map(user_to_index).values
columnas = df_score['product_id'].map(product_to_index).values

# Usar el score normalizado
valores = df_score['score_norm'].astype('float32').values

# Crear matriz dispersa usuario-producto
matriz_usuario_producto = csr_matrix(
    (valores, (filas, columnas)),
    shape=(len(usuarios_unicos), len(productos_unicos))
)

print("Dimensión de la matriz:", matriz_usuario_producto.shape)
print("Interacciones registradas:", matriz_usuario_producto.nnz)
print(f"Consumo de RAM aproximado: {matriz_usuario_producto.data.nbytes / (1024**2):.2f} MB")


Construyendo la Matriz Usuario-Producto...
Dimensión de la matriz: (1379296, 60525)
Interacciones registradas: 5682786
Consumo de RAM aproximado: 21.68 MB


In [7]:
# ==========================================================
# MUESTRA REPRESENTATIVA DE LA MATRIZ USUARIO-PRODUCTO
# ==========================================================

# Usuarios con más productos interactuados
usuarios_muestra = (
    df_score.groupby('user_id')['product_id']
    .nunique()
    .sort_values(ascending=False)
    .head(5)
    .index
)

# Productos más relevantes dentro de esos usuarios
productos_muestra = (
    df_score[df_score['user_id'].isin(usuarios_muestra)]
    .groupby('product_id')['score_norm']
    .sum()
    .sort_values(ascending=False)
    .head(8)
    .index
)

# Tabla usuario-producto pequeña
tabla_muestra = df_score[
    (df_score['user_id'].isin(usuarios_muestra)) &
    (df_score['product_id'].isin(productos_muestra))
].pivot_table(
    index='user_id',
    columns='product_id',
    values='score_norm',
    fill_value=0
)

# Renombrar para visualizar mejor
tabla_visual = tabla_muestra.copy()
tabla_visual.index = [f'usuario_{i+1}' for i in range(tabla_visual.shape[0])]
tabla_visual.columns = [f'producto_{i+1}' for i in range(tabla_visual.shape[1])]

tabla_visual

,producto_1,producto_2,producto_3,producto_4,producto_5,producto_6,producto_7,producto_8
usuario_1,0.000000,0.000000,0.000026,0.000026,0.000026,0.000026,0.000026,0.000026
usuario_2,0.000244,0.000279,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
usuario_3,0.000000,0.000000,0.000068,0.000068,0.000068,0.000068,0.000068,0.000068
usuario_4,0.000000,0.000000,0.000004,0.000004,0.000004,0.000004,0.000004,0.000004
usuario_5,0.000000,0.000000,0.000048,0.000048,0.000048,0.000048,0.000048,0.000048


## CONTENIDO

In [8]:
atributos_producto = df_ml.groupby('product_id', as_index=False).agg(
    category_code=('category_code', 'first'),
    brand=('brand', 'first'),
    price=('price', 'median')
)

atributos_producto.head()

,product_id,category_code,brand,price
0,1200216,apparel.shoes.slipons,apple,702.46
1,1200239,apparel.shoes.slipons,hp,288.15
2,1200519,apparel.shoes.slipons,elenberg,102.94
3,1200617,apparel.shoes.slipons,samsung,128.42
4,1200620,apparel.shoes.slipons,samsung,128.42


In [9]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from scipy.sparse import hstack, csr_matrix

encoder = OneHotEncoder(handle_unknown='ignore')

matriz_categoria_marca = encoder.fit_transform(
    atributos_producto[['category_code', 'brand']]
)

scaler = MinMaxScaler()

precio_norm = scaler.fit_transform(
    atributos_producto[['price']]
)

matriz_precio = csr_matrix(precio_norm)

matriz_contenido_producto = hstack(
    [matriz_categoria_marca, matriz_precio],
    format='csr'
)

print("Matriz producto-contenido:", matriz_contenido_producto.shape)

Matriz producto-contenido: (60525, 2117)


In [10]:
columnas = list(
    encoder.get_feature_names_out(['category_code', 'brand'])
) + ['price_norm']

muestra_contenido = pd.DataFrame(
    matriz_contenido_producto[:5, :10].toarray(),
    columns=columnas[:10]
)

muestra_contenido.index = [
    f'producto_{i+1}' for i in range(muestra_contenido.shape[0])
]

muestra_contenido

,category_code_accessories.bag,category_code_accessories.umbrella,category_code_accessories.wallet,category_code_apparel.belt,category_code_apparel.costume,category_code_apparel.dress,category_code_apparel.glove,category_code_apparel.jacket,category_code_apparel.jeans,category_code_apparel.jumper
producto_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
producto_2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
producto_3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
producto_4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
producto_5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# MODELADO

In [11]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd

modelo_knn = NearestNeighbors(
    metric='cosine',
    algorithm='brute',
    n_neighbors=6
)

modelo_knn.fit(matriz_usuario_producto)

print("Modelo KNN colaborativo entrenado.")

Modelo KNN colaborativo entrenado.


In [12]:
# ==========================================================
# 2. FILTRADO COLABORATIVO
# ==========================================================

import pandas as pd
import numpy as np

def recomendar_colaborativo(user_id, top_n=10):

    user_idx = user_to_index[user_id]

    distancias, indices = modelo_knn.kneighbors(
        matriz_usuario_producto[user_idx],
        n_neighbors=6
    )

    vecinos_idx = indices.flatten()[1:]
    similitudes = 1 - distancias.flatten()[1:]

    productos_usuario = set(
        df_score[df_score['user_id'] == user_id]['product_id']
    )

    candidatos = []

    for vecino_idx, similitud in zip(vecinos_idx, similitudes):

        vecino_id = usuarios_unicos[vecino_idx]

        productos_vecino = df_score[
            df_score['user_id'] == vecino_id
        ].copy()

        productos_vecino = productos_vecino[
            ~productos_vecino['product_id'].isin(productos_usuario)
        ]

        productos_vecino['score_colaborativo'] = (
            productos_vecino['score_norm'] * similitud
        )

        candidatos.append(
            productos_vecino[['product_id', 'score_colaborativo']]
        )

    recomendaciones = pd.concat(candidatos)

    recomendaciones = recomendaciones.groupby(
        'product_id',
        as_index=False
    )['score_colaborativo'].sum()

    return recomendaciones.sort_values(
        'score_colaborativo',
        ascending=False
    ).head(top_n)

In [13]:
# ==========================================================
# 3. FILTRADO BASADO EN CONTENIDO
# ==========================================================

from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix

productos_contenido = atributos_producto['product_id'].values

producto_a_idx_contenido = {
    product_id: idx
    for idx, product_id in enumerate(productos_contenido)
}


def crear_perfil_contenido_usuario(user_id):

    historial = df_score[df_score['user_id'] == user_id][
        ['product_id', 'score_norm']
    ]

    historial = historial[
        historial['product_id'].isin(producto_a_idx_contenido)
    ]

    indices_historial = historial['product_id'].map(
        producto_a_idx_contenido
    ).astype(int).values

    pesos = historial['score_norm'].values.reshape(-1, 1)

    perfil_usuario = matriz_contenido_producto[
        indices_historial
    ].multiply(pesos).sum(axis=0)

    return csr_matrix(perfil_usuario), indices_historial


def recomendar_contenido(user_id, top_n=10):

    perfil_usuario, indices_historial = crear_perfil_contenido_usuario(
        user_id
    )

    similitudes = cosine_similarity(
        perfil_usuario,
        matriz_contenido_producto
    ).ravel()

    similitudes[indices_historial] = -1

    top_indices = np.argsort(similitudes)[::-1][:top_n]

    recomendaciones = pd.DataFrame({
        'product_id': productos_contenido[top_indices],
        'score_contenido': similitudes[top_indices]
    })

    return recomendaciones.merge(
        atributos_producto[['product_id', 'category_code', 'brand', 'price']],
        on='product_id',
        how='left'
    )

In [14]:
# ==========================================================
# 4. MODELO HÍBRIDO ML
# 70% colaborativo + 30% contenido
# ==========================================================

from sklearn.preprocessing import MinMaxScaler

def recomendar_hibrido_ml(user_id, top_n=10):

    recomendaciones_cf = recomendar_colaborativo(
        user_id,
        top_n=top_n
    )

    recomendaciones_cf_detalle = recomendaciones_cf.merge(
        atributos_producto[['product_id', 'category_code', 'brand', 'price']],
        on='product_id',
        how='left'
    )

    recomendaciones_cb = recomendar_contenido(
        user_id,
        top_n=top_n
    )

    candidatos_ml = pd.concat([
        recomendaciones_cf_detalle[['product_id', 'score_colaborativo']],
        recomendaciones_cb[['product_id']].assign(score_colaborativo=0)
    ], ignore_index=True).drop_duplicates('product_id')

    perfil_usuario, _ = crear_perfil_contenido_usuario(user_id)

    candidatos_ml = candidatos_ml[
        candidatos_ml['product_id'].isin(producto_a_idx_contenido)
    ].copy()

    indices_candidatos = candidatos_ml['product_id'].map(
        producto_a_idx_contenido
    ).astype(int).values

    candidatos_ml['score_contenido'] = cosine_similarity(
        perfil_usuario,
        matriz_contenido_producto[indices_candidatos]
    ).ravel()

    scaler_cf = MinMaxScaler()
    candidatos_ml['score_colaborativo_norm'] = scaler_cf.fit_transform(
        candidatos_ml[['score_colaborativo']]
    )

    scaler_cb = MinMaxScaler()
    candidatos_ml['score_contenido_norm'] = scaler_cb.fit_transform(
        candidatos_ml[['score_contenido']]
    )

    candidatos_ml['score_ml_refinado'] = (
        0.7 * candidatos_ml['score_colaborativo_norm'] +
        0.3 * candidatos_ml['score_contenido_norm']
    )

    recomendaciones_ml_refinado = candidatos_ml.merge(
        atributos_producto[['product_id', 'category_code', 'brand', 'price']],
        on='product_id',
        how='left'
    ).sort_values(
        'score_ml_refinado',
        ascending=False
    ).head(top_n)

    return recomendaciones_ml_refinado

In [18]:
# ==========================================================
# 5. PRUEBA CON UN USUARIO
# ==========================================================

usuario_prueba = (
    df_score.groupby('user_id')['product_id']
    .nunique()
    .sort_values(ascending=False)
    .index[2]
)

recomendaciones_ml_refinado = recomendar_hibrido_ml(
    usuario_prueba,
    top_n=10
)

recomendaciones_ml_refinado

,product_id,score_colaborativo,score_contenido,score_colaborativo_norm,score_contenido_norm,score_ml_refinado,category_code,brand,price
0,100081779,1.562071e-07,0.311358,1.0,0.000000,0.700000,apparel.shoes,playmates,22.13
1,9400042,0.000000e+00,0.659496,0.0,1.000000,0.300000,apparel.scarf,unknown,944.43
2,100049836,0.000000e+00,0.632136,0.0,0.921411,0.276423,apparel.scarf,unknown,149.04
3,100074297,0.000000e+00,0.631026,0.0,0.918222,0.275467,apparel.scarf,unknown,128.68
4,100096833,0.000000e+00,0.631011,0.0,0.918181,0.275454,apparel.scarf,unknown,128.42
5,100229965,0.000000e+00,0.629684,0.0,0.914367,0.274310,apparel.shoes,unknown,1392.57
6,9200282,0.000000e+00,0.629580,0.0,0.914070,0.274221,apparel.scarf,unknown,102.71
7,100132193,0.000000e+00,0.629443,0.0,0.913676,0.274103,apparel.scarf,unknown,100.28
8,10700095,0.000000e+00,0.629262,0.0,0.913156,0.273947,apparel.scarf,unknown,97.07
9,9200540,0.000000e+00,0.629172,0.0,0.912896,0.273869,apparel.scarf,unknown,95.47


In [19]:
# ==========================================================
# 5. PRUEBA CON UN USUARIO Y DATOS DEL USUARIO
# ==========================================================

usuario_prueba = (
    df_score.groupby('user_id')['product_id']
    .nunique()
    .sort_values(ascending=False)
    .index[2]
)

print("Usuario probado:", usuario_prueba)

# Resumen del usuario
resumen_usuario = df_score[df_score['user_id'] == usuario_prueba].agg(
    productos_interactuados=('product_id', 'nunique'),
    score_total=('score_norm', 'sum'),
    score_promedio=('score_norm', 'mean')
)

display(resumen_usuario)

# Historial del usuario con detalle de productos
historial_usuario = df_score[
    df_score['user_id'] == usuario_prueba
].merge(
    atributos_producto[['product_id', 'category_code', 'brand', 'price']],
    on='product_id',
    how='left'
).sort_values(
    'score_norm',
    ascending=False
)

print("Historial principal del usuario:")
display(historial_usuario.head(10))

# Recomendaciones híbridas
recomendaciones_ml_refinado = recomendar_hibrido_ml(
    usuario_prueba,
    top_n=10
)

recomendaciones_ml_refinado.insert(
    0,
    'user_id',
    usuario_prueba
)

print("Recomendaciones híbridas para el usuario:")
display(recomendaciones_ml_refinado)

Usuario probado: 637360772


,product_id,score_norm
productos_interactuados,1991.0,NaN
score_total,NaN,0.005809
score_promedio,NaN,0.000003


Historial principal del usuario:


,user_id,product_id,peso_total,frecuencia,ultima_interaccion,dias_desde_ultima_interaccion,recencia,score,score_norm,category_code,brand,price
225,637360772,4100346,11,11,2020-04-09 10:17:45+00:00,21,0.045455,1.242453,0.000279,apparel.shoes,sony,653.81
213,637360772,4100277,10,10,2020-04-09 10:17:47+00:00,21,0.045455,1.089952,0.000244,apparel.shoes,sony,524.05
252,637360772,5100816,4,4,2020-04-07 13:18:27+00:00,23,0.041667,0.268240,0.000056,apparel.shoes,xiaomi,31.64
189,637360772,4100126,4,4,2020-04-07 13:19:07+00:00,23,0.041667,0.268240,0.000056,apparel.shoes,sony,383.02
1708,637360772,100166017,3,3,2020-04-08 10:24:57+00:00,22,0.043478,0.180821,0.000036,apparel.costume,sparta,15.06
1213,637360772,34200207,3,3,2020-04-08 09:49:02+00:00,22,0.043478,0.180821,0.000036,apparel.shoes.keds,boyscout,9.56
872,637360772,13000246,3,3,2020-04-08 13:06:57+00:00,22,0.043478,0.180821,0.000036,apparel.shorts,rainbo,90.07
1933,637360772,100193208,2,2,2020-04-08 04:15:58+00:00,22,0.043478,0.095532,0.000017,apparel.shoes.keds,garvalin,46.33
1762,637360772,100178677,2,2,2020-04-08 10:14:00+00:00,22,0.043478,0.095532,0.000017,apparel.shoes.sandals,unknown,32.26
1768,637360772,100181151,2,2,2020-04-08 10:13:12+00:00,22,0.043478,0.095532,0.000017,accessories.bag,loca,43.76


Recomendaciones híbridas para el usuario:


,user_id,product_id,score_colaborativo,score_contenido,score_colaborativo_norm,score_contenido_norm,score_ml_refinado,category_code,brand,price
0,637360772,100081779,1.562071e-07,0.311358,1.0,0.000000,0.700000,apparel.shoes,playmates,22.13
1,637360772,9400042,0.000000e+00,0.659496,0.0,1.000000,0.300000,apparel.scarf,unknown,944.43
2,637360772,100049836,0.000000e+00,0.632136,0.0,0.921411,0.276423,apparel.scarf,unknown,149.04
3,637360772,100074297,0.000000e+00,0.631026,0.0,0.918222,0.275467,apparel.scarf,unknown,128.68
4,637360772,100096833,0.000000e+00,0.631011,0.0,0.918181,0.275454,apparel.scarf,unknown,128.42
5,637360772,100229965,0.000000e+00,0.629684,0.0,0.914367,0.274310,apparel.shoes,unknown,1392.57
6,637360772,9200282,0.000000e+00,0.629580,0.0,0.914070,0.274221,apparel.scarf,unknown,102.71
7,637360772,100132193,0.000000e+00,0.629443,0.0,0.913676,0.274103,apparel.scarf,unknown,100.28
8,637360772,10700095,0.000000e+00,0.629262,0.0,0.913156,0.273947,apparel.scarf,unknown,97.07
9,637360772,9200540,0.000000e+00,0.629172,0.0,0.912896,0.273869,apparel.scarf,unknown,95.47
